# Homework: milestone 1, worked

Milestone 1 done on the worked example from
`sessions/session-11/solutions/worked-example-scope.md`, with
`data/clean/plays.csv` standing in for "my own file". Your steps and checks
will be different, because your data is. What should look the same is the
shape: logged steps, a list of problems, two routes to one number.

In [ ]:
import os
from pathlib import Path

import pandas as pd

here = Path.cwd()
while not (here / "data" / "clean" / "plays.csv").exists() and here != here.parent:
    here = here.parent
os.chdir(here)
print("working from:", Path.cwd())

In [ ]:
DATA_PATH = Path("data/clean/plays.csv")

raw = pd.read_csv(DATA_PATH)
print(f"{len(raw):,} rows read")

## Part 1: cleaning, one logged step at a time

In [ ]:
def logged(df, *steps):
    """Run each step on df in turn, print rows before and after each, return the result."""
    # *steps means "any number of arguments, collected into a tuple".
    # Each one is a function, and step.__name__ is that function's name as text.
    print(f"{'start':32} {len(df):>7,} rows")
    for step in steps:
        before = len(df)
        df = step(df)
        dropped = before - len(df)
        print(f"{step.__name__:32} {len(df):>7,} rows   ({dropped:,} dropped)")
    return df


def keep_needed_columns(df):
    """Keep only the columns the question uses, plus the id that identifies a play."""
    return df[["play_id", "played_at", "genre", "minutes_played"]].copy()


def parse_dates(df):
    """Turn played_at from text into real dates. Anything unreadable becomes NaT."""
    df = df.copy()
    df["played_at"] = pd.to_datetime(df["played_at"], format="%Y-%m-%d", errors="coerce")
    return df


def drop_incomplete_rows(df):
    """Drop rows missing a date, a genre or a number of minutes."""
    return df.dropna(subset=["played_at", "genre", "minutes_played"])


def keep_2024_and_2025(df):
    """Keep the two years the question compares, and add a year column."""
    df = df[df["played_at"].dt.year.isin([2024, 2025])].copy()
    df["year"] = df["played_at"].dt.year.astype(str)
    return df


clean = logged(raw, keep_needed_columns, parse_dates, drop_incomplete_rows, keep_2024_and_2025)

assert len(clean) == len(raw) == 2183

Nothing was dropped, by any step. On this file that is the expected
result, because it is the clean version of the log. But "nothing dropped"
is now something I **know**, step by step, rather than something I assume.
Run the same steps on a messier export next year, and the log will say
exactly where rows went.

Two choices worth noticing:

* `format="%Y-%m-%d"` with `errors="coerce"`: a date in any other format
  becomes missing instead of being guessed, and then `drop_incomplete_rows`
  counts it. Session 9 showed what a guess can do to day and month.
* the year column is text (`"2024"`), so the output columns are called
  `2024` and `2025` exactly as in notebook 01.

**The decisions, as they went into the worksheet:**

1. *Skipped plays are kept.* A skipped play still has the minutes it was
   actually played, and the question is about minutes listened. Dropping
   them would answer "minutes of songs I did not skip", which is a
   different question.
2. *Dates that are not `YYYY-MM-DD` are dropped, not guessed.* Here that
   is zero rows. If it were hundreds, I would have to fix the parsing
   instead, and say so.
3. *Minutes, not share.* Notebook 01 showed the biggest loser is
   Electronic by minutes and Folk by share of listening. The question says
   "listened to more", which is minutes. The share version is on the
   could list.

## Part 2: `find_problems`

In [ ]:
NEEDED = ["play_id", "played_at", "genre", "minutes_played"]


def find_problems(df):
    """Return a list of problems with the cleaned data, as sentences. Empty means fine."""
    problems = []

    missing_columns = [c for c in NEEDED if c not in df.columns]
    if missing_columns:
        problems.append(f"missing columns: {missing_columns}")
        return problems              # the other checks need the columns, so stop here

    for column in NEEDED:
        n = df[column].isna().sum()
        if n:
            problems.append(f"{n} missing values in {column}")

    if not pd.api.types.is_numeric_dtype(df["minutes_played"]):
        problems.append("minutes_played is not a number column")
    else:
        out_of_range = ((df["minutes_played"] <= 0) | (df["minutes_played"] > 60)).sum()
        if out_of_range:
            problems.append(f"{out_of_range} plays outside 0 to 60 minutes")

    dates = pd.to_datetime(df["played_at"], format="%Y-%m-%d", errors="coerce")
    unreadable = dates.isna().sum() - df["played_at"].isna().sum()
    if unreadable:
        problems.append(f"{unreadable} dates not in YYYY-MM-DD form")
    months = dates.dt.strftime("%Y-%m").nunique()
    if months != 24:
        problems.append(f"dates cover {months} months, not the 24 the question needs")

    duplicated_ids = df["play_id"].duplicated().sum()
    if duplicated_ids:
        problems.append(f"{duplicated_ids} duplicated play_id values")

    return problems


problems = find_problems(clean)
print(problems)
assert not problems, problems

Empty list: nothing wrong. But a check that never fails proves nothing
until you have seen it fail. So here it is on the messy file from session
9, with only the badly named column fixed, so that the other checks get a
chance to run.

In [ ]:
messy = pd.read_csv("data/messy/plays_messy.csv")
print(find_problems(messy))

In [ ]:
messy = messy.rename(columns={"minutes played ": "minutes_played"})

for problem in find_problems(messy):
    print("-", problem)

First run: it stops at the missing column, because `minutes played `
(with spaces) is not `minutes_played`. Second run, with that one thing
renamed: five problems, listed at once, each with a count. 36 missing
genres, 94 missing minutes, minutes stored as text, 430 dates in another
format, 40 duplicates. That is why `find_problems` returns a list instead
of using `assert` on each line: you see the whole picture before you start
fixing.

Notice what it does **not** find: `phone` versus `PHONE`, or genres with
stray spaces. Nothing here asks about them, because the worked example
does not use the device column and only uses genre as a label. A check
only finds what you thought to ask about. If your question depends on a
category being spelled one way, write a check for that too.

It is also why the session 9 cleaning happens **before** the checks: the
checks describe what clean data looks like, and the cleaning makes it so.

In [ ]:
messy_problems = find_problems(messy)
assert len(messy_problems) == 5
assert "40 duplicated play_id values" in messy_problems

## Part 3: the must output, checked two ways

In [ ]:
def analyse(df):
    """Return minutes per genre per year, with the change, smallest change first."""
    result = df.groupby(["genre", "year"])["minutes_played"].sum().unstack("year").round(1)
    result["change"] = (result["2025"] - result["2024"]).round(1)
    return result.sort_values("change")


result = analyse(clean)
result

In [ ]:
# Route 1: the parts add up to the whole (the table rounded each cell, so allow a little).
assert abs(result[["2024", "2025"]].sum().sum() - clean["minutes_played"].sum()) < 0.5

# Route 2: one cell worked out with a filter instead of a groupby.
folk_2024 = clean[(clean["genre"] == "Folk") & (clean["year"] == "2024")]["minutes_played"].sum()
assert round(folk_2024, 1) == result.loc["Folk", "2024"] == 832.3

print("both routes agree")

In [ ]:
out = Path("output") / "session-11" / "genre_change.csv"
out.parent.mkdir(parents=True, exist_ok=True)
result.to_csv(out)
print(f"wrote {out}")

## Part 4: into `project/pipeline.py`

For the worked example, the pipeline's `clean` became one line:

```python
def clean(df):
    """Clean the plays, printing one line per step."""
    return logged(df, keep_needed_columns, parse_dates,
                  drop_incomplete_rows, keep_2024_and_2025)
```

with the three decisions above listed in its docstring, the way the
session 10 version lists its own. `main()` gained the two lines that
refuse to write an output from data that fails a check, straight after
`data = clean(raw)`:

```python
problems = find_problems(data)
assert not problems, problems
```

and `check`, commented out at the first run, came back with a job that
fits this table: read `summary.csv` back and compare the sum of the 2024
and 2025 columns with the total minutes in the cleaned data.

Running `python project/pipeline.py` then printed the cleaning log, found
no problems, and wrote the same eight-row table to
`project/output/summary.csv`.
Commit message: `Milestone 1: logged cleaning, data checks, genre change table`.

**What makes this milestone 1 and not just "some code":** if the data
changes, the pipeline either produces a trustworthy answer or stops and
says what is wrong. Everything in milestones 2 and 3 (the chart, the
README, the presentation) is built on top of that, so it is worth having
first.